## Preprocessing

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 42
FORCE_REPROCESS = True

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

DATASET_NAMES = [
    "ml-latest-small",
    "ml-1m",
    "ml-latest",
]

In [3]:
def get_paths(dataset_name):
    raw_dir = DATA_DIR / dataset_name
    processed_dir = DATA_DIR / "processed" / dataset_name
    return raw_dir, processed_dir

In [4]:
def required_processed_files(dataset_name):
    files = [
        "train_ratings.csv",
        "test_ratings.csv",
        "movies.csv",
        "movies_with_idx.csv",
        "movie_map.csv",
        "user_map.csv",
    ]

    if dataset_name in {"ml-latest-small", "ml-1m"}:
        files += [
            "R_train.npy",
            "train_mask.npy",
        ]

    return files

In [5]:
def processed_exists(dataset_name):
    _, processed_dir = get_paths(dataset_name)

    return processed_dir.exists() and all(
        (processed_dir / file_name).exists()
        for file_name in required_processed_files(dataset_name)
    )

In [6]:
def load_movielens_raw(dataset_name, raw_dir):
    if dataset_name == "ml-1m":
        ratings = pd.read_csv(
            raw_dir / "ratings.dat",
            sep="::",
            names=["userId", "movieId", "rating", "timestamp"],
            engine="python",
            encoding="latin-1",
        )

        movies = pd.read_csv(
            raw_dir / "movies.dat",
            sep="::",
            names=["movieId", "title", "genres"],
            engine="python",
            encoding="latin-1",
        )

    else:
        ratings = pd.read_csv(raw_dir / "ratings.csv")
        movies = pd.read_csv(raw_dir / "movies.csv")

    return ratings, movies

In [7]:
def build_rating_matrix(df, n_users, n_movies):
    R = np.zeros((n_users, n_movies), dtype=np.float64)

    user_indices = df["user_idx"].to_numpy()
    movie_indices = df["movie_idx"].to_numpy()
    rating_values = df["rating"].to_numpy()

    R[user_indices, movie_indices] = rating_values

    return R

In [ ]:
def preprocess_dataset(dataset_name, force=False):
    raw_dir, processed_dir = get_paths(dataset_name)

    if processed_exists(dataset_name) and not force:
        print(f"{dataset_name}: processed data already exists. Skipping.")
        return

    print(f"\nProcessing {dataset_name}...")
    print("Raw dir:", raw_dir)
    print("Processed dir:", processed_dir)

    processed_dir.mkdir(parents=True, exist_ok=True)

    ratings, movies = load_movielens_raw(dataset_name, raw_dir)

    train_df, test_df = train_test_split(
        ratings,
        test_size=0.2,
        random_state=SEED,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    user_ids = sorted(ratings["userId"].unique())
    movie_ids = sorted(ratings["movieId"].unique())

    user_to_idx = {
        user_id: idx
        for idx, user_id in enumerate(user_ids)
    }

    movie_to_idx = {
        movie_id: idx
        for idx, movie_id in enumerate(movie_ids)
    }

    def add_matrix_indices(df):
        df = df.copy()
        df["user_idx"] = df["userId"].map(user_to_idx).astype("int32")
        df["movie_idx"] = df["movieId"].map(movie_to_idx).astype("int32")
        return df

    train_df = add_matrix_indices(train_df)
    test_df = add_matrix_indices(test_df)

    n_users = len(user_to_idx)
    n_movies = len(movie_to_idx)

    train_df.to_csv(processed_dir / "train_ratings.csv", index=False)
    test_df.to_csv(processed_dir / "test_ratings.csv", index=False)

    movies.to_csv(processed_dir / "movies.csv", index=False)

    movie_map = pd.DataFrame({
        "movieId": list(movie_to_idx.keys()),
        "movie_idx": list(movie_to_idx.values()),
    })

    user_map = pd.DataFrame({
        "userId": list(user_to_idx.keys()),
        "user_idx": list(user_to_idx.values()),
    })

    movie_map.to_csv(processed_dir / "movie_map.csv", index=False)
    user_map.to_csv(processed_dir / "user_map.csv", index=False)

    movies_with_idx = movies.copy()
    movies_with_idx["movie_idx"] = movies_with_idx["movieId"].map(movie_to_idx)
    movies_with_idx = movies_with_idx.dropna(subset=["movie_idx"])
    movies_with_idx["movie_idx"] = movies_with_idx["movie_idx"].astype("int32")

    movies_with_idx.to_csv(processed_dir / "movies_with_idx.csv", index=False)

    if dataset_name in {"ml-latest-small", "ml-1m"}:
        R_train = build_rating_matrix(
            train_df,
            n_users=n_users,
            n_movies=n_movies,
        )

        train_mask = R_train > 0

        np.save(processed_dir / "R_train.npy", R_train)
        np.save(processed_dir / "train_mask.npy", train_mask)

        print("R_train shape:", R_train.shape)
        print("Observed train ratings:", train_mask.sum())
    else:
        print("Dense R_train skipped for ml-latest.")

    print(f"{dataset_name}: preprocessing done.")

In [9]:
for dataset_name in DATASET_NAMES:
    preprocess_dataset(dataset_name, force=FORCE_REPROCESS)


Processing ml-latest-small...
Raw dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\ml-latest-small
Processed dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\processed\ml-latest-small
R_train shape: (610, 9724)
Observed train ratings: 80668
ml-latest-small: preprocessing done.

Processing ml-1m...
Raw dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\ml-1m
Processed dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\processed\ml-1m
R_train shape: (6040, 3706)
Observed train ratings: 800167
ml-1m: preprocessing done.

Processing ml-latest...
Raw dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\ml-latest
Processed dir: c:\Users\Labea.DESKTOP-6B3LI5E\OneDrive\Dokumente\GitHub\Data-Lab\data\processed\ml-latest
Dense R_train skipped for ml-latest.
ml-latest: preprocessing done.
